# 대화 상태 (Conversation State)
모델 상호작용 중 대화 상태를 관리하는 방법을 알아보세요.
OpenAI는 대화 상태를 관리하는 몇 가지 방법을 제공하며, 이는 대화에서 여러 메시지 또는 차례에 걸쳐 정보를 보존하는 데 중요합니다.

각 텍스트 생성 요청은 독립적이고 상태가 지정되지 않지만(Assistants API를 사용하는 경우 제외), 텍스트 생성 요청에 추가 메시지를 매개변수로 제공하여 여러 차례에 걸친 대화를 구현할 수 있습니다. 

### 컨텍스트 창 관리
컨텍스트 창을 이해하면 스레드된 대화를 성공적으로 생성하고 모델 상호작용 전반의 상태를 관리하는 데 도움이 됩니다.   
컨텍스트 창은 단일 요청에서 사용할 수 있는 최대 토큰 수입니다. 이 최대 토큰 수에는 입력, 출력 및 추론 토큰이 포함됩니다. 모델의 컨텍스트 창에 대한 자세한 내용은 모델 세부 정보를 참조하세요. 

다음 토큰 수가 컨텍스트 창 총계에 적용됩니다.
- 입력 토큰( Responses APIinput 에 대한 배열 에 포함하는 입력)  
- 출력 토큰(프롬프트에 대한 응답으로 생성된 토큰)  
- 추론 토큰(모델이 응답을 계획하는 데 사용됨)
  
컨텍스트 창 제한을 초과하여 생성된 토큰은 API 응답에서 잘릴 수 있습니다.

<img src="https://i.imgur.com/WcWPlmZ.png" width=500 />

### 대화 상태 관리
response API를 사용하면 대화 상태를 자동으로 더 쉽게 관리할 수 있으므로 대화가 진행될 때마다 수동으로 입력을 전달할 필요가 없습니다.  
Responses API는 상태를 저장하므로 대화 전반의 컨텍스트 관리가 간단한 매개변수로 가능합니다.

### 문서/텍스트를 기반으로 챗봇이 답변하도록 하기

짧은 안내 문서(FAQ)를 챗봇에게 넘겨주고, `previous_response_id` 로 대화 맥락을 이어가는 **간단한 예제**입니다.

핵심은 두 가지입니다.
1. `instructions` 로 챗봇의 역할과 규칙을 정한다.
2. 참고할 문서를 입력으로 넘겨주면, 챗봇이 그 문서 범위 안에서만 답한다.

In [ ]:
# 챗봇이 참고할 짧은 안내 문서 (FAQ)

In [ ]:
# instructions 로 역할/규칙을 정하고, 첫 입력으로 안내 문서를 넘겨줍니다.
# previous_response_id 로 이전 대화 맥락을 이어갑니다.

------------------
(대화 예시)
```
assistant: 안녕하세요! 카페 무브입니다. 무엇을 도와드릴까요?
사용자 입력 (종료: quit):  영업시간이 어떻게 되나요?
assistant: 카페 무브는 매일 오전 9시부터 오후 10시까지 연중무휴로 운영합니다.
사용자 입력 (종료: quit):  주차도 되나요?
assistant: 네, 1시간 무료 주차가 가능하며 이후에는 10분당 1,000원입니다.
사용자 입력 (종료: quit):  강아지 데려가도 되나요?
assistant: 소형견에 한해 동반 입장이 가능합니다.
사용자 입력 (종료: quit):  배달도 되나요?
assistant: 죄송합니다. 해당 정보는 없습니다.
사용자 입력 (종료: quit):  quit
```

> `previous_response_id` 덕분에 "주차도 되나요?", "강아지 데려가도 되나요?" 처럼 짧게 물어도 이전 맥락(카페 무브)을 기억하고 답합니다.

## 스트리밍으로 응답 받기 (Streaming)

기본적으로 OpenAI API에 요청을 보내면 모델이 **전체 출력을 다 생성한 후** 단일 HTTP 응답으로 한 번에 돌려줍니다. 긴 답변을 생성할 경우 사용자는 응답을 기다리는 동안 빈 화면을 보게 됩니다.

스트리밍 응답을 사용하면 모델이 출력을 생성하는 도중에 **시작 부분부터 조각(chunk) 단위로** 받아서 바로 처리하거나 화면에 출력할 수 있습니다.

응답 스트리밍을 시작하려면 `client.responses.create(...)` 요청에 `stream=True` 를 설정하면 됩니다.

> 스트리밍은 특히 **대화가 오가는 챗봇** 맥락에서 그 효과가 가장 잘 드러납니다. 토큰이 실시간으로 찍히는 경험이 응답 대기 체감 시간을 크게 줄여 주기 때문입니다. 아래에서 앞서 다룬 대화 맥락 관리와 스트리밍을 함께 적용해 봅니다.

In [ ]:
# 응답이 조각(chunk) 단위로 도착할 때마다 출력

### 스트리밍 + 대화 맥락을 함께 적용한 챗봇

이제 앞에서 배운 **대화 맥락 유지**와 방금 본 **스트리밍**을 하나로 합쳐 봅니다.

- 대화 히스토리를 `list` 로 직접 관리하면서 (`conversation_history`)
- 매 응답을 `stream=True` 로 받아 **토큰이 생성되는 즉시 화면에 출력**합니다.

이렇게 하면 사용자는 응답이 완성될 때까지 기다리지 않고, 글자가 실시간으로 찍히는 자연스러운 챗봇 경험을 하게 됩니다.

In [ ]:
# 대화 히스토리를 list로 관리하면서, 응답을 스트리밍으로 출력하는 챗봇
# - input 에 conversation_history(list) 를 통째로 넘겨 맥락을 유지합니다.
# - stream=True 로 토큰이 생성되는 즉시 화면에 출력합니다.
    # 사용자 입력을 히스토리에 추가
    # 지금까지의 대화 전체를 input 으로 넘기고, 스트리밍으로 응답 받기
    # 완성된 응답을 히스토리에 추가하여 다음 차례의 맥락으로 사용

### 실습 문제: 대화 흐름과 상태 유지

`OpenAI Responses API`의 `previous_response_id` 를 사용하여, **이전 응답을 기억하고 연관성 있게 대화를 이어가는** 디지털 헬스 상담 챗봇을 만들어 보세요.

### 주어진 초기 설정

- 사용자 첫 질문: `"요즘 소화가 잘 안돼요. 어떻게 해야 할까요?"`
- 이어지는 질문: `"그럼 매운 음식은 피해야 하나요?"`

### 구현 조건

1. Responses API를 사용하여 첫 번째 질문에 답변을 생성하세요.
2. `previous_response_id` 를 활용하여 두 번째 질문을 **동일한 컨텍스트 흐름**으로 이어서 응답하도록 하세요.
3. 두 번째 응답은 **첫 번째 대답을 기억한 상태**에서 이어지는 조언처럼 들려야 합니다.
4. 응답 마지막에 `"다른 증상은 없으신가요?"` 와 같이 자연스럽게 다음 질문을 유도하세요.

### 테스트 입력 예시

- 첫 질문: `"요즘 소화가 잘 안돼요. 어떻게 해야 할까요?"`
- 후속: `"그럼 매운 음식은 피해야 하나요?"` → 앞의 소화 맥락을 이어 매운 음식 조언